# 07 — Mission-Critical Infrastructure
### Supply chain, CVEs, reproducibility, air-gap

---

**Format:** Narrated demo — pre-run outputs. The pipeline doesn't change. The infrastructure around it does.

**The question this module answers:** Can you prove the environment your pipeline runs in is safe, reproducible, and deployable without internet access?

```
Layer    Tool                               What it answers
───────  ─────────────────────────────────  ──────────────────────────────────────────
Lock     conda-lock                         Is this environment bit-for-bit reproducible?
Scan     anaconda-audit                     Does it contain known vulnerabilities?
Gate     Anaconda Platform policy filter    Did anything vulnerable get in upstream?
Pack     conda-pack                         Can we deploy without internet access?
Verify   AIBOM + SHA-256                    Is the model file what we think it is?
```

None of these require pipeline code changes. Zero. The `HarnessedLightcurveFlow` from Module 06 is untouched.

---
## Part 1 — Lock: from spec to contract

`environment.yml` is a specification — it says what you want. `conda-lock` produces a contract — it says exactly what you'll get, down to the build hash, on every platform.

The difference matters six months from now when a package releases a new version that breaks something. With a lock file, the environment that passed your CVE scan is the environment that runs in production.

In [ ]:
# conda-lock lock -f ../06-app-architecture/environment.yml -p linux-64
#
# Produces: conda-linux-64.lock
# Install from lock: conda-lock install --name app-architecture-locked conda-linux-64.lock

print("""environment.yml (specification — what you want):
  - python=3.11
  - polars>=1.0        ← any 1.x release
  - metaflow>=2.18     ← any 2.18+
  - duckdb>=0.10

conda-lock.yml (contract — exactly what you get):
  - polars==1.26.0=py311h4e23daf_0  (linux-64, conda-forge)
  - metaflow==2.18.4=py311h4e23daf_0
  - duckdb==0.10.3=py311h4e23daf_0
  ... 247 more entries, every transitive dependency pinned

SHA-256 of lock file: a3f2c1d4e5b6...
Commit this. It is your deployment contract.""")

environment.yml (specification — what you want):
  - python=3.11
  - polars>=1.0        ← any 1.x release
  - metaflow>=2.18     ← any 2.18+
  - duckdb>=0.10

conda-lock.yml (contract — exactly what you get):
  - polars==1.26.0=py311h4e23daf_0  (linux-64, conda-forge)
  - metaflow==2.18.4=py311h4e23daf_0
  - duckdb==0.10.3=py311h4e23daf_0
  ... 247 more entries, every transitive dependency pinned

SHA-256 of lock file: a3f2c1d4e5b6...
Commit this. It is your deployment contract.


The lock file is also what `scripts/lock_and_scan.sh` feeds into the CVE scan. You're scanning the exact environment that will deploy, not a floating spec.

---
## Part 2 — Scan: anaconda-audit

`anaconda-audit` scans a conda environment against CVE data that Anaconda pulls from NVD/NIST. It goes further than a raw NVD lookup — Anaconda curates each CVE against their package builds, so you get a status (Active, Cleared, Mitigated, Disputed) rather than just a score.

**Installation (in base environment — installs three conda plugins):**
```bash
conda install --name base anaconda-cloud::anaconda-env-manager
```
Installs: `anaconda-audit` (scan), `anaconda-env-log` (auto-log environments to Platform), `anaconda-activate-check` (block activation of non-compliant envs).

In [ ]:
# anaconda audit scan --name app-architecture-locked
#
# Key outputs:
#   ✓CVE-...  — checkmark means Anaconda has curated this CVE
#   Cleared   — Anaconda reviewed it, not applicable to this build
#   Active    — exploitable, you should act
#   Mitigated — Anaconda patched it in this build

print("""$ anaconda audit scan --name app-architecture-locked

Package              Version  Build             Channel       CVE              Score  Status
────────────────────────────────────────────────────────────────────────────────────────────────
polars               1.26.0   py311h4e23daf_0   conda-forge   —                —      Clean
metaflow             2.18.4   py311h4e23daf_0   conda-forge   —                —      Clean
duckdb               0.10.3   py311h4e23daf_0   conda-forge   —                —      Clean
openssl              3.0.18   h5eee18b_0        defaults      ✓CVE-2024-0727   2.0    Cleared
openssl              3.0.18   h5eee18b_0        defaults      CVE-2023-5678    9.8    Active  ⚠
pydantic             2.12.4   py311h4e23daf_0   conda-forge   —                —      Clean
numpy                2.3.5    py311hb5e798b_0   conda-forge   —                —      Clean
...

Summary:
  Severity    Total   Active  Cleared  Mitigated  Disputed  Reported
  Critical    1       1       0        0          0         0
  High        0       0       0        0          0         0
  Medium      0       0       0        0          0         0
  Low         0       0       0        0          0         0
  None        1       0       1        0          0         0

  ⚠  1 active critical CVE found. See CVE-2023-5678 in openssl.""")

$ anaconda audit scan --name app-architecture-locked

Package              Version  Build             Channel       CVE              Score  Status
────────────────────────────────────────────────────────────────────────────────────────────────
polars               1.26.0   py311h4e23daf_0   conda-forge   —                —      Clean
metaflow             2.18.4   py311h4e23daf_0   conda-forge   —                —      Clean
duckdb               0.10.3   py311h4e23daf_0   conda-forge   —                —      Clean
openssl              3.0.18   h5eee18b_0        defaults      ✓CVE-2024-0727   2.0    Cleared
openssl              3.0.18   h5eee18b_0        defaults      CVE-2023-5678    9.8    Active  ⚠
pydantic             2.12.4   py311h4e23daf_0   conda-forge   —                —      Clean
numpy                2.3.5    py311hb5e798b_0   conda-forge   —                —      Clean
...

Summary:
  Severity    Total   Active  Cleared  Mitigated  Disputed  Reported
  Critical    1       1

In [ ]:
# Remediation path when a critical CVE is found:
# 1. Upgrade the package (check Anaconda Platform CVE channel — updated every 4h)
# 2. Re-lock after upgrading
# 3. Re-scan the new lock

print("""Resolution: upgrade openssl to a version where Anaconda marks it mitigated or cleared.

$ conda update --name app-architecture-locked openssl
  Updating openssl 3.0.18 → 3.3.4  (CVE-2023-5678 cleared in this build)

$ anaconda audit scan --name app-architecture-locked | grep openssl
  openssl  3.3.4  py311h...  conda-forge  ✓CVE-2023-5678  9.8  Cleared
                              ✓CVE-2024-0727  2.0  Cleared

✓  Re-lock after upgrading:
   conda-lock lock -f environment.yml -p linux-64

The Anaconda curation mark (✓) means Anaconda reviewed this CVE
against the specific build, not just the package version.
That's the difference between NVD raw data and Anaconda's curated feed.""")

Resolution: upgrade openssl to a version where Anaconda marks it mitigated or cleared.

$ conda update --name app-architecture-locked openssl
  Updating openssl 3.0.18 → 3.3.4  (CVE-2023-5678 cleared in this build)

$ anaconda audit scan --name app-architecture-locked | grep openssl
  openssl  3.3.4  py311h...  conda-forge  ✓CVE-2023-5678  9.8  Cleared
                                           ✓CVE-2024-0727  2.0  Cleared

✓  Re-lock after upgrading:
   conda-lock lock -f environment.yml -p linux-64

The Anaconda curation mark (✓) means Anaconda reviewed this CVE
against the specific build, not just the package version.
That's the difference between NVD raw data and Anaconda's curated feed.


---
## Part 3 — Gate: Anaconda Platform policy filter

`anaconda-audit` catches CVEs after the fact — you scan an environment you've already built.

A **policy filter** catches them before packages ever reach your developers. It runs upstream: when the channel mirror pulls from conda-forge or Anaconda main, packages that fail the policy are excluded. Developers working from your internal channel can't install a blocked package because it isn't there.

This is the Anaconda Platform admin capability. Policy filters are the supply chain gate that `anaconda-audit` is the local-scanner complement to.

In [ ]:
# The policy is defined in security/policy.yaml
# Applied via Anaconda Platform UI: Policies → Create Policy
# Then attached to the channel mirror: Channels → [channel] → Create Mirror → Assign Policy

print("""Policy: ai-pipeline-production

CVE Rules (from Anaconda recommended defaults):
  ALLOW if: CVE score < 7.0
         OR CVE status in (mitigated, cleared)

  This means:
    openssl 3.0.18 with CVE score 9.8 (Active)   → BLOCKED
    openssl 3.3.4  with CVE score 9.8 (Cleared)  → ALLOWED
    requests 2.31  with CVE score 5.0 (Active)   → ALLOWED (score < 7)

Package Rules:
  Platform:       linux-64, linux-aarch64, osx-arm64
  License:        MIT, BSD, Apache-2.0, PSF, Python-2.0, ISC
  Only signed:    True  (Anaconda signature required)

Mirror Frequency: weekly (Sunday 02:00 UTC)
CVE channel sync: every 4 hours (daily for air-gapped installs)

When the mirror runs:
  - Policy is applied to every package in the source channel
  - Packages that fail CVE or license rules are excluded
  - Developers working from this internal channel can't install blocked packages
  - The gate runs before anyone types 'conda install'""")

Policy: ai-pipeline-production

CVE Rules (from Anaconda recommended defaults):
  ALLOW if: CVE score < 7.0
         OR CVE status in (mitigated, cleared)

  This means:
    openssl 3.0.18 with CVE score 9.8 (Active)   → BLOCKED
    openssl 3.3.4  with CVE score 9.8 (Cleared)  → ALLOWED
    requests 2.31  with CVE score 5.0 (Active)   → ALLOWED (score < 7)

Package Rules:
  Platform:       linux-64, linux-aarch64, osx-arm64
  License:        MIT, BSD, Apache-2.0, PSF, Python-2.0, ISC
  Only signed:    True  (Anaconda signature required)

Mirror Frequency: weekly (Sunday 02:00 UTC)
CVE channel sync: every 4 hours (daily for air-gapped installs)

When the mirror runs:
  - Policy is applied to every package in the source channel
  - Packages that fail CVE or license rules are excluded
  - Developers working from this internal channel can't install blocked packages
  - The gate runs before anyone types 'conda install'


**The relationship between policy and `anaconda-audit`:**

Policy filter = **upstream gate** (blocks at the channel, before install)  
`anaconda-audit` = **downstream check** (confirms after install, catches drift)

You need both. Policy prevents most problems. `anaconda-audit` catches what slips through — packages installed before the policy ran, or CVEs newly disclosed after the last mirror sync.

With both in place, `scripts/lock_and_scan.sh` becomes the CI step that proves the environment passed both gates before deploying.

---
## Part 4 — Pack: conda-pack for air-gapped deployment

`conda-pack` takes a conda environment and produces a relocatable tarball. The target machine doesn't need conda installed, doesn't need internet access, and doesn't need to be able to reach Anaconda Platform.

This is the physical artifact for air-gapped deployment. The entire stack ships in one file.

In [ ]:
# conda-pack --name app-architecture-locked --output app-architecture.tar.gz

print("""$ conda-pack --name app-architecture-locked --output app-architecture.tar.gz
Collecting packages...
Packing environment at '/opt/conda/envs/app-architecture-locked' to 'app-architecture.tar.gz'
[########################################] | 100% Completed |  2min 14.3s

Tarball size: 847MB
Contains: Python 3.11, all conda packages, compiled binaries

What's inside:
  /bin/python3.11
  /lib/python3.11/site-packages/polars/
  /lib/python3.11/site-packages/metaflow/
  /lib/python3.11/site-packages/duckdb/
  /lib/libssl.so.3    ← the openssl we verified above
  ... 247 packages, all at the locked versions

What's NOT inside:
  NVIDIA GPU drivers (host-level — must be installed on target)
  System glibc (conda-pack preserves everything above it)""")

$ conda-pack --name app-architecture-locked --output app-architecture.tar.gz
Packing environment at '/opt/conda/envs/app-architecture-locked' to 'app-architecture.tar.gz'
[########################################] | 100% Completed |  2min 14.3s

Tarball size: 847MB
Contains: Python 3.11, all conda packages, compiled binaries

What's inside:
  /bin/python3.11
  /lib/python3.11/site-packages/polars/
  /lib/python3.11/site-packages/metaflow/
  /lib/python3.11/site-packages/duckdb/
  /lib/libssl.so.3    ← the openssl we verified above
  ... 247 packages, all at the locked versions

What's NOT inside:
  NVIDIA GPU drivers (host-level — must be installed on target)
  System glibc (conda-pack preserves everything above it)


In [ ]:
# Extraction and activation on the air-gapped target (from scripts/pack_and_ship.sh)

print("""# On the air-gapped target machine (no conda, no internet required):

mkdir -p /opt/lightcurve-pipeline/env
tar -xzf app-architecture.tar.gz -C /opt/lightcurve-pipeline/env/

# One-time path fix (conda-pack requirement):
source /opt/lightcurve-pipeline/env/bin/activate
conda-unpack

# Run the pipeline — no internet, no conda, no Platform:
source /opt/lightcurve-pipeline/env/bin/activate
python flows/harnessed_lightcurve_flow.py run --targets wasp18b

# The DuckDB memory store travels with it:
# /opt/lightcurve-pipeline/memory/lightcurve_memory.duckdb
# Agent memory from development carries into production.""")

# On the air-gapped target machine (no conda, no internet required):

mkdir -p /opt/lightcurve-pipeline/env
tar -xzf app-architecture.tar.gz -C /opt/lightcurve-pipeline/env/

# One-time path fix (conda-pack requirement):
source /opt/lightcurve-pipeline/env/bin/activate
conda-unpack

# Run the pipeline — no internet, no conda, no Platform:
source /opt/lightcurve-pipeline/env/bin/activate
python flows/harnessed_lightcurve_flow.py run --targets wasp18b

# The DuckDB memory store travels with it:
# /opt/lightcurve-pipeline/memory/lightcurve_memory.duckdb
# Agent memory from development carries into production.


---
## Part 5 — Verify: AIBOM + SHA-256 model provenance

The pipeline environment is verified. What about the model it calls?

Anaconda Platform's Model Catalog provides an **AIBOM** (AI Bill of Materials) for every model: a CycloneDX JSON file containing SHA-256 checksums per quantization, benchmark scores, software dependencies, and ethical considerations. Download it from the model's Overview tab.

Before deploying any model, verify the file you downloaded matches the AIBOM checksum. A mismatch means the file was corrupted in transit or replaced.

In [ ]:
# python security/verify_aibom.py --aibom model.aibom.json --model models/model.gguf
#
# The AIBOM is downloaded from Anaconda Platform Model Catalog:
#   Model Catalog → [model] → Overview tab → Download AIBOM

print("""$ python security/verify_aibom.py \\
    --aibom mistral-7b.aibom.json \\
    --model models/mistral-7b-instruct-q4_k_m.gguf

Loading AIBOM: mistral-7b.aibom.json
  Found 6 hash entries in AIBOM

  Verifying mistral-7b-instruct-q4_k_m.gguf...
  ✓  SHA-256 match: a3f2c1d4e5b6f7a8...

── Verification Summary ─────────────────────────────────────────
  AIBOM entries:   6
  Verified:        1
  Failed:          0
  No AIBOM entry:  0

  ✓  All files verified successfully.
─────────────────────────────────────────────────────────────────""")

$ python security/verify_aibom.py \
    --aibom mistral-7b.aibom.json \
    --model models/mistral-7b-instruct-q4_k_m.gguf

Loading AIBOM: mistral-7b.aibom.json
  Found 6 hash entries in AIBOM

  Verifying mistral-7b-instruct-q4_k_m.gguf...
  ✓  SHA-256 match: a3f2c1d4e5b6f7a8...

── Verification Summary ─────────────────────────────────────────
  AIBOM entries:   6
  Verified:        1
  Failed:          0
  No AIBOM entry:  0

  ✓  All files verified successfully.
─────────────────────────────────────────────────────────────────


In [ ]:
# The AIBOM is more than just a checksum — it's the model's provenance record.
# This is what makes Anaconda Platform's model governance auditable.

print("""What else is in the AIBOM (beyond hashes):

  Model metadata:
    publisher: MistralAI
    architecture: Transformer, decoder-only
    license: Apache-2.0
    intended_use: text generation, instruction following

  Performance (from Anaconda benchmark runs):
    HellaSwag:  81.4 / 100
    WinoGrande: 76.2 / 100
    TruthfulQA: 52.1 / 100

  Ethical considerations:
    Known limitations: May hallucinate on low-frequency topics
    Bias risks: Training data bias toward English-language content
    Recommended mitigations: Validate outputs in production; use RAG for factual queries

  Software dependencies:
    torch>=2.1, transformers>=4.35, sentencepiece

  File variants:
    Q4_K_M: 4.37GB, max_ram=6.0GB  ← verified above
    Q8_0:   7.17GB, max_ram=10.0GB
    F16:    14.48GB, max_ram=18.0GB""")

What else is in the AIBOM (beyond hashes):

  Model metadata:
    publisher: MistralAI
    architecture: Transformer, decoder-only
    license: Apache-2.0
    intended_use: text generation, instruction following

  Performance (from Anaconda benchmark runs):
    HellaSwag:  81.4 / 100
    WinoGrande: 76.2 / 100
    TruthfulQA: 52.1 / 100

  Ethical considerations:
    Known limitations: May hallucinate on low-frequency topics
    Bias risks: Training data bias toward English-language content
    Recommended mitigations: Validate outputs in production; use RAG for factual queries

  Software dependencies:
    torch>=2.1, transformers>=4.35, sentencepiece

  File variants:
    Q4_K_M: 4.37GB, max_ram=6.0GB  ← verified above
    Q8_0:   7.17GB, max_ram=10.0GB
    F16:    14.48GB, max_ram=18.0GB


---
## The complete supply chain

```
Anaconda Platform (admin)
  ↓  Policy filter: CVE < 7 OR mitigated/cleared, OSI license, signed only
  ↓  Mirror runs weekly → internal channel updated

Developer machine
  ↓  conda install from internal channel (policy-filtered)
  ↓  conda-lock → pinned lock file committed to repo
  ↓  anaconda-audit scan → CVE check on locked environment
  ↓  scripts/lock_and_scan.sh → CI gate passes

Deployment artifact
  ↓  conda-pack → relocatable tarball (847MB)
  ↓  verify_aibom.py → model file SHA-256 verified
  ↓  scripts/pack_and_ship.sh → tarball + memory store shipped

Air-gapped production node
  ↓  tar -xzf + conda-unpack (no internet required)
  ↓  python flows/harnessed_lightcurve_flow.py run
     @conda per step → each step's environment independently locked and scanned
```

The `@conda` decorator from Module 03 means each Metaflow step has its own environment. That means the CVE scan and lock apply per step — a vulnerability in the analyze step's LLM client doesn't create a surface in the ingest step's Polars environment. The supply chain argument made in Module 03 is what makes this module's CI gate granular.

---

**The pipeline code** — `HarnessedLightcurveFlow`, `ingestion.py`, `evals/assertions.py`, `vectordb/memory_store.py` — **is unchanged from Module 06.** Supply chain security is infrastructure, not application code.